<a href="https://colab.research.google.com/github/megamiro-code/battlefield/blob/main/%E5%85%B5%E5%A3%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from IPython.display import HTML, display

html = r"""
<iframe
    style="width:100%; height:820px; border:0; background:#000;"
    srcdoc='
<!DOCTYPE html>
<html lang="ja">
<head>
<meta charset="utf-8">

<style>
html, body {
    margin: 0;
    padding: 0;
    overflow: hidden;
    background: #000;
    font-family: Arial, sans-serif;
}

#info {
    position: absolute;
    top: 10px;
    left: 10px;
    z-index: 10;

    color: white;
    background: rgba(0,0,0,0.72);

    padding: 10px 12px;
    border-radius: 6px;

    font-size: 13px;
    line-height: 1.45;

    min-width: 240px;
}

#message {
    position: absolute;
    left: 50%;
    top: 50%;

    transform: translate(-50%, -50%);

    z-index: 20;

    color: white;
    background: rgba(0,0,0,0.80);

    padding: 20px 30px;

    border-radius: 10px;

    font-size: 28px;

    display: none;

    text-align: center;
}
</style>
</head>

<body>

<div id="info">
    FPS: <span id="fps">0</span><br>
    Time: <span id="time">0.0</span> s<br>
    Remaining: <span id="remaining">180.0</span> s<br>

    Red alive: <span id="redAlive">100</span><br>
    Blue alive: <span id="blueAlive">100</span><br>

    Red commander HP: <span id="redCommanderHP">1.00</span><br>
    Blue commander HP: <span id="blueCommanderHP">1.00</span><br>

    Attacks: <span id="attacks">0</span><br>
    Blocked moves: <span id="blocked">0</span><br>
    Stuck units: <span id="stuck">0</span><br>
    Overlap corrections: <span id="overlap">0</span><br>

    NN command updates: <span id="commandUpdates">0</span><br>
    NN forward passes: <span id="forwardPasses">0</span><br>

    NN input size: <span id="inputSize">0</span><br>
    NN output size: <span id="outputSize">0</span><br>

    Red reward: <span id="redReward">0.000</span><br>
    Blue reward: <span id="blueReward">0.000</span><br>
</div>

<div id="message"></div>

<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/build/three.min.js"></script>

<script>

/* =========================================================
   CONFIGURATION
   ========================================================= */

const FIELD_SIZE = 10.0;
const HALF_FIELD = FIELD_SIZE / 2.0;

const RED = 0;
const BLUE = 1;

const SOLDIER_COUNT = 100;
const TOTAL_UNITS = 202;

/*
 * 制限時間
 *
 * 以前 30秒
 *      ↓
 * 今回 180秒
 */
const MAX_BATTLE_TIME = 180.0;


/* -------------------------
   NN command
   ------------------------- */

const COMMAND_INTERVAL = 0.20;

const TERRAIN_RESOLUTION = 10;

const TERRAIN_FEATURES =
    TERRAIN_RESOLUTION * TERRAIN_RESOLUTION;

const UNIT_FEATURES =
    TOTAL_UNITS * 10;

const INPUT_SIZE =
    TERRAIN_FEATURES + UNIT_FEATURES;

/*
 * 100 soldiers ×
 *
 *   dx
 *   dz
 *   attack
 *
 * = 300 outputs
 */
const OUTPUT_SIZE = SOLDIER_COUNT * 3;

const HIDDEN1 = 256;
const HIDDEN2 = 128;


/* -------------------------
   Movement
   ------------------------- */

const COMMANDER_SPEED = 0.20;

const SOLDIER_SPEED_MIN = 0.40;
const SOLDIER_SPEED_MAX = 0.65;


/* -------------------------
   Combat
   ------------------------- */

const ATTACK_RANGE = 0.42;

const ATTACK_COOLDOWN = 0.55;

const ATTACK_DAMAGE = 0.20;


/* -------------------------
   Collision
   ------------------------- */

const SOLDIER_RADIUS = 0.13;

const COMMANDER_RADIUS = 0.22;

const COMMANDER_EXTRA_MARGIN = 0.10;

const OVERLAP_ITERATIONS = 3;


/* =========================================================
   THREE.JS
   ========================================================= */

const scene = new THREE.Scene();

scene.background =
    new THREE.Color(0x111111);


const camera =
    new THREE.PerspectiveCamera(
        45,
        window.innerWidth / window.innerHeight,
        0.1,
        1000
    );

camera.position.set(
    0,
    15,
    5
);

camera.lookAt(
    0,
    0,
    0
);


const renderer =
    new THREE.WebGLRenderer({
        antialias: true
    });

renderer.setSize(
    window.innerWidth,
    window.innerHeight
);

renderer.setPixelRatio(
    Math.min(window.devicePixelRatio, 2)
);

document.body.appendChild(
    renderer.domElement
);


/* =========================================================
   LIGHT
   ========================================================= */

scene.add(
    new THREE.AmbientLight(
        0xffffff,
        0.75
    )
);

const directionalLight =
    new THREE.DirectionalLight(
        0xffffff,
        0.8
    );

directionalLight.position.set(
    5,
    10,
    5
);

scene.add(
    directionalLight
);


/* =========================================================
   FIELD
   ========================================================= */

const fieldGeometry =
    new THREE.PlaneGeometry(
        FIELD_SIZE,
        FIELD_SIZE
    );

const fieldMaterial =
    new THREE.MeshStandardMaterial({
        color: 0x243024
    });

const field =
    new THREE.Mesh(
        fieldGeometry,
        fieldMaterial
    );

field.rotation.x =
    -Math.PI / 2;

scene.add(field);


/* =========================================================
   GRID
   ========================================================= */

const gridMaterial =
    new THREE.LineBasicMaterial({
        color: 0x4f594f
    });

for (let i = 0; i <= TERRAIN_RESOLUTION; i++) {

    const p =
        -HALF_FIELD + i;

    const gx =
        new THREE.BufferGeometry().setFromPoints([
            new THREE.Vector3(
                p,
                0.012,
                -HALF_FIELD
            ),
            new THREE.Vector3(
                p,
                0.012,
                HALF_FIELD
            )
        ]);

    scene.add(
        new THREE.Line(
            gx,
            gridMaterial
        )
    );


    const gz =
        new THREE.BufferGeometry().setFromPoints([
            new THREE.Vector3(
                -HALF_FIELD,
                0.012,
                p
            ),
            new THREE.Vector3(
                HALF_FIELD,
                0.012,
                p
            )
        ]);

    scene.add(
        new THREE.Line(
            gz,
            gridMaterial
        )
    );
}


/* =========================================================
   WALLS
   ========================================================= */

const walls = [
    { x:-4.5, z:-4.5 },
    { x:-4.5, z: 4.5 },
    { x: 4.5, z:-4.5 },
    { x: 4.5, z: 4.5 },

    { x:-0.5, z:-0.5 },
    { x:-0.5, z: 0.5 },
    { x: 0.5, z:-0.5 },
    { x: 0.5, z: 0.5 }
];


const wallMaterial =
    new THREE.MeshStandardMaterial({
        color: 0x555555
    });

for (const wall of walls) {

    const geometry =
        new THREE.BoxGeometry(
            1,
            0.7,
            1
        );

    const mesh =
        new THREE.Mesh(
            geometry,
            wallMaterial
        );

    mesh.position.set(
        wall.x,
        0.35,
        wall.z
    );

    scene.add(mesh);
}


/* =========================================================
   TERRAIN MAP
   ========================================================= */

const terrainMap =
    new Array(
        TERRAIN_RESOLUTION *
        TERRAIN_RESOLUTION
    ).fill(0);


function terrainIndex(ix, iz) {

    return iz *
        TERRAIN_RESOLUTION +
        ix;
}


for (
    let iz = 0;
    iz < TERRAIN_RESOLUTION;
    iz++
) {

    for (
        let ix = 0;
        ix < TERRAIN_RESOLUTION;
        ix++
    ) {

        const cx =
            -HALF_FIELD +
            ix +
            0.5;

        const cz =
            -HALF_FIELD +
            iz +
            0.5;

        let blocked = false;

        for (const wall of walls) {

            if (
                Math.abs(cx - wall.x) < 0.5 &&
                Math.abs(cz - wall.z) < 0.5
            ) {
                blocked = true;
                break;
            }
        }

        terrainMap[
            terrainIndex(ix, iz)
        ] =
            blocked ? 1 : 0;
    }
}


/* =========================================================
   RANDOM
   ========================================================= */

function randomRange(a, b) {

    return a +
        Math.random() *
        (b - a);
}


function randomNormal() {

    let u = 0;
    let v = 0;

    while (u === 0)
        u = Math.random();

    while (v === 0)
        v = Math.random();

    return Math.sqrt(
        -2.0 *
        Math.log(u)
    ) *
    Math.cos(
        2.0 *
        Math.PI *
        v
    );
}


/* =========================================================
   NN
   ========================================================= */

function makeMatrix(rows, cols) {

    const m =
        new Array(rows);

    const scale =
        Math.sqrt(
            2.0 /
            (rows + cols)
        );

    for (let r = 0; r < rows; r++) {

        m[r] =
            new Float32Array(cols);

        for (let c = 0; c < cols; c++) {

            m[r][c] =
                randomNormal() *
                scale;
        }
    }

    return m;
}


function makeVector(n) {

    return new Float32Array(n);
}


class CommanderNetwork {

    constructor() {

        this.W1 =
            makeMatrix(
                HIDDEN1,
                INPUT_SIZE
            );

        this.b1 =
            makeVector(
                HIDDEN1
            );

        this.W2 =
            makeMatrix(
                HIDDEN2,
                HIDDEN1
            );

        this.b2 =
            makeVector(
                HIDDEN2
            );

        this.W3 =
            makeMatrix(
                OUTPUT_SIZE,
                HIDDEN2
            );

        this.b3 =
            makeVector(
                OUTPUT_SIZE
            );
    }


    forward(input) {

        /*
         * Layer 1
         */

        const h1 =
            new Float32Array(
                HIDDEN1
            );

        for (
            let i = 0;
            i < HIDDEN1;
            i++
        ) {

            let sum =
                this.b1[i];

            const row =
                this.W1[i];

            for (
                let j = 0;
                j < INPUT_SIZE;
                j++
            ) {

                sum +=
                    row[j] *
                    input[j];
            }

            h1[i] =
                sum > 0 ?
                sum :
                0;
        }


        /*
         * Layer 2
         */

        const h2 =
            new Float32Array(
                HIDDEN2
            );

        for (
            let i = 0;
            i < HIDDEN2;
            i++
        ) {

            let sum =
                this.b2[i];

            const row =
                this.W2[i];

            for (
                let j = 0;
                j < HIDDEN1;
                j++
            ) {

                sum +=
                    row[j] *
                    h1[j];
            }

            h2[i] =
                sum > 0 ?
                sum :
                0;
        }


        /*
         * Output
         */

        const out =
            new Float32Array(
                OUTPUT_SIZE
            );

        for (
            let i = 0;
            i < OUTPUT_SIZE;
            i++
        ) {

            let sum =
                this.b3[i];

            const row =
                this.W3[i];

            for (
                let j = 0;
                j < HIDDEN2;
                j++
            ) {

                sum +=
                    row[j] *
                    h2[j];
            }

            out[i] =
                sum;
        }

        return out;
    }
}


const redNetwork =
    new CommanderNetwork();

const blueNetwork =
    new CommanderNetwork();


/* =========================================================
   UNIT DATA
   ========================================================= */

const units = [];


/*
 * unit structure:
 *
 * {
 *   id,
 *   team,
 *   commander,
 *   x,
 *   z,
 *   vx,
 *   vz,
 *   hp,
 *   maxHp,
 *   speed,
 *   alive,
 *   attackCommand,
 *   attackTimer,
 *   commandDx,
 *   commandDz,
 *   mesh,
 *   body,
 *   head,
 *   crown,
 *   stuckTime
 * }
 */


/* =========================================================
   COLLISION HELPERS
   ========================================================= */

function unitRadius(unit) {

    return unit.commander ?
        COMMANDER_RADIUS :
        SOLDIER_RADIUS;
}


function circlesOverlap(
    ax,
    az,
    ar,
    bx,
    bz,
    br
) {

    const dx =
        bx - ax;

    const dz =
        bz - az;

    const d2 =
        dx * dx +
        dz * dz;

    const required =
        ar +
        br +
        (
            unitRadiusValueExtra(
                ax,
                az,
                bx,
                bz,
                ar,
                br
            )
        );

    return d2 <
        required * required;
}


/*
 * Commander gets an additional
 * separation margin.
 *
 * This is intentionally deterministic.
 */
function unitRadiusValueExtra(
    ax,
    az,
    bx,
    bz,
    ar,
    br
) {

    return 0;
}


function distanceSquared(
    a,
    b
) {

    const dx =
        a.x - b.x;

    const dz =
        a.z - b.z;

    return dx * dx + dz * dz;
}


/* =========================================================
   WALL COLLISION
   ========================================================= */

function positionHitsWall(
    x,
    z,
    radius
) {

    for (const wall of walls) {

        const minX =
            wall.x - 0.5;

        const maxX =
            wall.x + 0.5;

        const minZ =
            wall.z - 0.5;

        const maxZ =
            wall.z + 0.5;

        const closestX =
            Math.max(
                minX,
                Math.min(x, maxX)
            );

        const closestZ =
            Math.max(
                minZ,
                Math.min(z, maxZ)
            );

        const dx =
            x - closestX;

        const dz =
            z - closestZ;

        if (
            dx * dx +
            dz * dz
            <
            radius * radius
        ) {
            return true;
        }
    }

    return false;
}


function insideField(
    x,
    z,
    radius
) {

    return (
        x >=
        -HALF_FIELD + radius
        &&
        x <=
        HALF_FIELD - radius
        &&
        z >=
        -HALF_FIELD + radius
        &&
        z <=
        HALF_FIELD - radius
    );
}


function positionAllowed(
    unit,
    x,
    z
) {

    const radius =
        unitRadius(unit);

    if (
        !insideField(
            x,
            z,
            radius
        )
    ) {
        return false;
    }

    if (
        positionHitsWall(
            x,
            z,
            radius
        )
    ) {
        return false;
    }

    return true;
}


function canOccupy(
    unit,
    x,
    z
) {

    if (
        !positionAllowed(
            unit,
            x,
            z
        )
    ) {
        return false;
    }


    for (const other of units) {

        if (other === unit)
            continue;

        if (!other.alive)
            continue;

        const dx =
            x - other.x;

        const dz =
            z - other.z;

        const d =
            Math.sqrt(
                dx * dx +
                dz * dz
            );

        let required =
            unitRadius(unit) +
            unitRadius(other);

        if (
            unit.commander ||
            other.commander
        ) {
            required +=
                COMMANDER_EXTRA_MARGIN;
        }

        if (d < required) {
            return false;
        }
    }

    return true;
}


/* =========================================================
   POSITION INITIALIZATION
   ========================================================= */

function randomValidPosition(
    unit,
    minX,
    maxX
) {

    for (let attempt = 0; attempt < 5000; attempt++) {

        const x =
            randomRange(
                minX,
                maxX
            );

        const z =
            randomRange(
                -4.3,
                4.3
            );

        if (
            canOccupy(
                unit,
                x,
                z
            )
        ) {
            return {
                x,
                z
            };
        }
    }

    /*
     * Fallback.
     */
    return {
        x: minX,
        z: 0
    };
}


/* =========================================================
   CREATE SOLDIER MODEL
   ========================================================= */

function createSoldierMesh(
    unit
) {

    const group =
        new THREE.Group();

    const isRed =
        unit.team === RED;

    const mainColor =
        isRed ?
        0xdd4444 :
        0x4477ee;


    /*
     * Body
     */

    const bodyGeometry =
        new THREE.BoxGeometry(
            0.24,
            0.32,
            0.24
        );

    const bodyMaterial =
        new THREE.MeshStandardMaterial({
            color: mainColor
        });

    const body =
        new THREE.Mesh(
            bodyGeometry,
            bodyMaterial
        );

    body.position.y =
        0.22;

    group.add(body);


    /*
     * Head
     */

    const headGeometry =
        new THREE.BoxGeometry(
            0.20,
            0.20,
            0.20
        );

    const headMaterial =
        new THREE.MeshStandardMaterial({
            color: 0xe0c39b
        });

    const head =
        new THREE.Mesh(
            headGeometry,
            headMaterial
        );

    head.position.y =
        0.48;

    group.add(head);


    unit.mesh =
        group;

    unit.body =
        body;

    unit.head =
        head;

    group.position.set(
        unit.x,
        0,
        unit.z
    );

    scene.add(group);
}


/* =========================================================
   CREATE COMMANDER MODEL
   ========================================================= */

function createCommanderMesh(
    unit
) {

    const group =
        new THREE.Group();

    const isRed =
        unit.team === RED;

    const mainColor =
        isRed ?
        0xcc2222 :
        0x2255cc;


    const bodyGeometry =
        new THREE.BoxGeometry(
            0.42,
            0.55,
            0.42
        );

    const bodyMaterial =
        new THREE.MeshStandardMaterial({
            color: mainColor
        });

    const body =
        new THREE.Mesh(
            bodyGeometry,
            bodyMaterial
        );

    body.position.y =
        0.35;

    group.add(body);


    const headGeometry =
        new THREE.BoxGeometry(
            0.32,
            0.30,
            0.32
        );

    const headMaterial =
        new THREE.MeshStandardMaterial({
            color: 0xe0c39b
        });

    const head =
        new THREE.Mesh(
            headGeometry,
            headMaterial
        );

    head.position.y =
        0.78;

    group.add(head);


    /*
     * Crown
     */

    const crownGeometry =
        new THREE.ConeGeometry(
            0.20,
            0.20,
            5
        );

    const crownMaterial =
        new THREE.MeshStandardMaterial({
            color: 0xffd700
        });

    const crown =
        new THREE.Mesh(
            crownGeometry,
            crownMaterial
        );

    crown.rotation.x =
        Math.PI;

    crown.position.y =
        1.08;

    group.add(crown);


    unit.mesh =
        group;

    unit.body =
        body;

    unit.head =
        head;

    unit.crown =
        crown;

    group.position.set(
        unit.x,
        0,
        unit.z
    );

    scene.add(group);
}


/* =========================================================
   UNIT CREATION
   ========================================================= */

function createUnit(
    team,
    commander,
    x,
    z
) {

    const unit = {

        id: units.length,

        team: team,

        commander: commander,

        x: x,
        z: z,

        vx: 0,
        vz: 0,

        hp: commander ? 1.0 : 1.0,
        maxHp: commander ? 1.0 : 1.0,

        speed:
            commander
            ? COMMANDER_SPEED
            : randomRange(
                SOLDIER_SPEED_MIN,
                SOLDIER_SPEED_MAX
            ),

        alive: true,

        attackCommand: false,

        attackTimer: 0,

        commandDx: 0,
        commandDz: 0,

        stuckTime: 0,

        mesh: null,
        body: null,
        head: null,
        crown: null
    };

    units.push(unit);

    if (commander) {
        createCommanderMesh(unit);
    }
    else {
        createSoldierMesh(unit);
    }

    return unit;
}


/* =========================================================
   INITIAL UNITS
   ========================================================= */

/*
 * IMPORTANT:
 * commanders are created FIRST.
 *
 * This avoids soldiers being placed
 * inside a commander.
 */

const redCommander =
    createUnit(
        RED,
        true,
        -3.2,
        0
    );

const blueCommander =
    createUnit(
        BLUE,
        true,
        3.2,
        0
    );


/*
 * Red soldiers
 */

for (
    let i = 0;
    i < SOLDIER_COUNT;
    i++
) {

    const dummy = {
        commander: false,
        team: RED,
        x: 0,
        z: 0
    };

    const p =
        randomValidPosition(
            dummy,
            -4.3,
            -0.8
        );

    createUnit(
        RED,
        false,
        p.x,
        p.z
    );
}


/*
 * Blue soldiers
 */

for (
    let i = 0;
    i < SOLDIER_COUNT;
    i++
) {

    const dummy = {
        commander: false,
        team: BLUE,
        x: 0,
        z: 0
    };

    const p =
        randomValidPosition(
            dummy,
            0.8,
            4.3
        );

    createUnit(
        BLUE,
        false,
        p.x,
        p.z
    );
}


/* =========================================================
   NN INPUT
   ========================================================= */

function buildObservation(
    perspectiveTeam
) {

    const input =
        new Float32Array(
            INPUT_SIZE
        );

    let index = 0;


    /*
     * Terrain
     *
     * 0 = walkable
     * 1 = wall
     */

    for (let i = 0; i < terrainMap.length; i++) {

        input[index++] =
            terrainMap[i];
    }


    /*
     * Unit features:
     *
     * x
     * z
     * vx
     * vz
     * HP
     * self-team
     * enemy-team
     * soldier
     * commander
     * alive
     */

    for (const unit of units) {

        input[index++] =
            unit.x / HALF_FIELD;

        input[index++] =
            unit.z / HALF_FIELD;

        input[index++] =
            unit.vx;

        input[index++] =
            unit.vz;

        input[index++] =
            unit.hp;

        const own =
            unit.team === perspectiveTeam;

        input[index++] =
            own ? 1 : 0;

        input[index++] =
            own ? 0 : 1;

        input[index++] =
            unit.commander ? 0 : 1;

        input[index++] =
            unit.commander ? 1 : 0;

        input[index++] =
            unit.alive ? 1 : 0;
    }

    return input;
}


/* =========================================================
   ACTION DECODING
   ========================================================= */

function sigmoid(x) {

    if (x >= 0) {

        const z =
            Math.exp(-x);

        return 1 /
            (1 + z);
    }

    const z =
        Math.exp(x);

    return z /
        (1 + z);
}


function applyNetworkOutput(
    output,
    perspectiveTeam
) {

    /*
     * Output:
     *
     * soldier 0:
     *   [dx, dz, attack]
     *
     * soldier 1:
     *   [dx, dz, attack]
     *
     * ...
     */

    let soldierIndex = 0;

    for (const unit of units) {

        if (
            unit.commander ||
            unit.team !== perspectiveTeam
        ) {
            continue;
        }

        const base =
            soldierIndex * 3;

        let dx =
            Math.tanh(
                output[base]
            );

        let dz =
            Math.tanh(
                output[base + 1]
            );

        /*
         * Normalize movement command.
         *
         * This prevents a diagonal command from
         * becoming faster than a straight command.
         */

        const length =
            Math.sqrt(
                dx * dx +
                dz * dz
            );

        if (length > 1e-6) {

            dx /= length;
            dz /= length;
        }

        unit.commandDx = dx;
        unit.commandDz = dz;

        unit.attackCommand =
            sigmoid(
                output[base + 2]
            ) > 0.5;

        soldierIndex++;
    }
}


/* =========================================================
   NN COMMAND UPDATE
   ========================================================= */

let commandUpdateCount = 0;
let forwardPassCount = 0;

function updateCommands() {

    const redInput =
        buildObservation(
            RED
        );

    const blueInput =
        buildObservation(
            BLUE
        );


    /*
     * Two independent networks.
     */

    const redOutput =
        redNetwork.forward(
            redInput
        );

    forwardPassCount++;


    const blueOutput =
        blueNetwork.forward(
            blueInput
        );

    forwardPassCount++;


    applyNetworkOutput(
        redOutput,
        RED
    );

    applyNetworkOutput(
        blueOutput,
        BLUE
    );


    commandUpdateCount++;
}


/* =========================================================
   MOVEMENT
   ========================================================= */

let blockedMoves = 0;
let overlapCorrections = 0;


function tryMove(
    unit,
    nx,
    nz
) {

    if (
        canOccupy(
            unit,
            nx,
            nz
        )
    ) {

        unit.x = nx;
        unit.z = nz;

        return true;
    }

    return false;
}


function executeSoldierMovement(
    unit,
    dt
) {

    if (!unit.alive)
        return;


    const dx =
        unit.commandDx;

    const dz =
        unit.commandDz;


    /*
     * No artificial random side movement.
     *
     * AI command is obeyed as-is,
     * subject only to the physical rules.
     */

    if (
        Math.abs(dx) < 1e-8 &&
        Math.abs(dz) < 1e-8
    ) {

        unit.vx = 0;
        unit.vz = 0;

        return;
    }


    const distance =
        unit.speed * dt;

    const moveX =
        dx * distance;

    const moveZ =
        dz * distance;


    const nx =
        unit.x + moveX;

    const nz =
        unit.z + moveZ;


    /*
     * Direct movement
     */

    if (
        tryMove(
            unit,
            nx,
            nz
        )
    ) {

        unit.vx = dx;
        unit.vz = dz;

        return;
    }


    /*
     * Try x-only.
     */

    if (
        tryMove(
            unit,
            unit.x + moveX,
            unit.z
        )
    ) {

        unit.vx = dx;
        unit.vz = 0;

        return;
    }


    /*
     * Try z-only.
     */

    if (
        tryMove(
            unit,
            unit.x,
            unit.z + moveZ
        )
    ) {

        unit.vx = 0;
        unit.vz = dz;

        return;
    }


    /*
     * Try tangent directions.
     *
     * These are not random.
     *
     * They simply represent directions
     * allowed by the collision geometry.
     */

    const candidates = [

        [ -dz,  dx ],
        [  dz, -dx ],

    ];


    for (
        const dir of candidates
    ) {

        const tx =
            dir[0];

        const tz =
            dir[1];

        if (
            tryMove(
                unit,
                unit.x + tx * distance,
                unit.z + tz * distance
            )
        ) {

            unit.vx = tx;
            unit.vz = tz;

            return;
        }
    }


    /*
     * Completely blocked.
     */

    unit.vx = 0;
    unit.vz = 0;

    blockedMoves++;
}


/* =========================================================
   COMMANDER MOVEMENT
   ========================================================= */

/*
 * The on-field commander is currently not controlled
 * by the soldier-output head.
 *
 * It uses its current movement state.
 *
 * This can later be integrated into the policy as an
 * additional action.
 */

function updateCommanderMovement(
    unit,
    dt
) {

    if (!unit.alive)
        return;

    /*
     * Current prototype:
     * commander remains at its current position.
     *
     * The strategic AI controls the soldiers.
     *
     * This can later be changed to an explicit
     * commander action output.
     */

    unit.vx = 0;
    unit.vz = 0;
}


/* =========================================================
   OVERLAP RESOLUTION
   ========================================================= */

function resolveOverlaps() {

    for (
        let iteration = 0;
        iteration < OVERLAP_ITERATIONS;
        iteration++
    ) {

        for (
            let i = 0;
            i < units.length;
            i++
        ) {

            const a =
                units[i];

            if (!a.alive)
                continue;


            for (
                let j = i + 1;
                j < units.length;
                j++
            ) {

                const b =
                    units[j];

                if (!b.alive)
                    continue;


                let dx =
                    b.x - a.x;

                let dz =
                    b.z - a.z;

                let d2 =
                    dx * dx +
                    dz * dz;


                let d;

                if (d2 < 1e-12) {

                    /*
                     * Deterministic direction.
                     *
                     * No random lateral movement.
                     */

                    const angle =
                        (
                            (
                                i * 92821 +
                                j * 68917 +
                                iteration * 17
                            ) % 360
                        ) *
                        Math.PI /
                        180;

                    dx =
                        Math.cos(angle);

                    dz =
                        Math.sin(angle);

                    d = 1;

                } else {

                    d =
                        Math.sqrt(d2);
                }


                let required =
                    unitRadius(a) +
                    unitRadius(b);

                if (
                    a.commander ||
                    b.commander
                ) {
                    required +=
                        COMMANDER_EXTRA_MARGIN;
                }


                if (
                    d < required
                ) {

                    const overlap =
                        required - d;

                    const nx =
                        dx / d;

                    const nz =
                        dz / d;


                    /*
                     * Commander receives less displacement.
                     */

                    if (
                        a.commander &&
                        !b.commander
                    ) {

                        const push =
                            overlap;

                        b.x +=
                            nx * push;

                        b.z +=
                            nz * push;

                    } else if (
                        b.commander &&
                        !a.commander
                    ) {

                        const push =
                            overlap;

                        a.x -=
                            nx * push;

                        a.z -=
                            nz * push;

                    } else {

                        const half =
                            overlap * 0.5;

                        a.x -=
                            nx * half;

                        a.z -=
                            nz * half;

                        b.x +=
                            nx * half;

                        b.z +=
                            nz * half;
                    }


                    /*
                     * Keep inside field.
                     */

                    if (!a.commander) {

                        a.x =
                            Math.max(
                                -HALF_FIELD + SOLDIER_RADIUS,
                                Math.min(
                                    HALF_FIELD - SOLDIER_RADIUS,
                                    a.x
                                )
                            );

                        a.z =
                            Math.max(
                                -HALF_FIELD + SOLDIER_RADIUS,
                                Math.min(
                                    HALF_FIELD - SOLDIER_RADIUS,
                                    a.z
                                )
                            );
                    }


                    if (!b.commander) {

                        b.x =
                            Math.max(
                                -HALF_FIELD + SOLDIER_RADIUS,
                                Math.min(
                                    HALF_FIELD - SOLDIER_RADIUS,
                                    b.x
                                )
                            );

                        b.z =
                            Math.max(
                                -HALF_FIELD + SOLDIER_RADIUS,
                                Math.min(
                                    HALF_FIELD - SOLDIER_RADIUS,
                                    b.z
                                )
                            );
                    }


                    overlapCorrections++;
                }
            }
        }
    }
}


/* =========================================================
   COMBAT
   ========================================================= */

let attackCount = 0;


function findAttackTarget(
    attacker
) {

    let target = null;

    let bestDistance2 =
        Infinity;


    for (const enemy of units) {

        if (!enemy.alive)
            continue;

        if (
            enemy.team ===
            attacker.team
        ) {
            continue;
        }


        const dx =
            enemy.x -
            attacker.x;

        const dz =
            enemy.z -
            attacker.z;

        const d2 =
            dx * dx +
            dz * dz;


        if (
            d2 <=
            ATTACK_RANGE *
            ATTACK_RANGE
        ) {

            if (
                d2 <
                bestDistance2
            ) {

                bestDistance2 =
                    d2;

                target =
                    enemy;
            }
        }
    }


    return target;
}


function performAttacks(
    dt
) {

    for (const unit of units) {

        if (!unit.alive)
            continue;

        if (unit.commander)
            continue;


        unit.attackTimer =
            Math.max(
                0,
                unit.attackTimer - dt
            );


        /*
         * AI must explicitly choose attack.
         */

        if (!unit.attackCommand)
            continue;


        /*
         * Attack only when cooldown is finished.
         */

        if (
            unit.attackTimer > 0
        ) {
            continue;
        }


        const target =
            findAttackTarget(
                unit
            );

        if (!target)
            continue;


        /*
         * Attack causes hard stun.
         *
         * The movement system checks attackTimer.
         * The actual cooldown is set here.
         */

        unit.attackTimer =
            ATTACK_COOLDOWN;

        target.hp -=
            ATTACK_DAMAGE;

        attackCount++;


        if (
            target.hp <= 0
        ) {

            target.hp = 0;
            target.alive = false;

            if (
                target.mesh
            ) {

                target.mesh.visible =
                    false;
            }


            /*
             * Commander death ends battle immediately.
             */

            if (
                target.commander
            ) {

                endBattle(
                    unit.team === RED
                        ? "RED"
                        : "BLUE",
                    "commander"
                );

                return;
            }
        }
    }
}


/* =========================================================
   MOVEMENT + HARD STUN
   ========================================================= */

function updateMovement(
    dt
) {

    for (const unit of units) {

        if (!unit.alive)
            continue;


        /*
         * Commanders currently do not move.
         */

        if (unit.commander) {

            updateCommanderMovement(
                unit,
                dt
            );

            continue;
        }


        /*
         * Attack cooldown represents
         * attack hard-stun.
         */

        if (
            unit.attackTimer > 0
        ) {

            unit.vx = 0;
            unit.vz = 0;

            continue;
        }


        executeSoldierMovement(
            unit,
            dt
        );
    }
}


/* =========================================================
   STUCK DETECTION
   ========================================================= */

let stuckCount = 0;


function updateStuckDetection(
    dt
) {

    stuckCount = 0;

    for (const unit of units) {

        if (!unit.alive)
            continue;

        const moving =
            Math.abs(unit.vx) +
            Math.abs(unit.vz) >
            0.01;


        const commandMagnitude =
            Math.abs(unit.commandDx) +
            Math.abs(unit.commandDz);


        /*
         * Only count as stuck if:
         *
         *   command exists
         *   but actual movement is zero
         */

        if (
            commandMagnitude > 0.05 &&
            !moving &&
            unit.attackTimer <= 0
        ) {

            unit.stuckTime += dt;

        } else {

            unit.stuckTime = 0;
        }


        if (
            unit.stuckTime >= 0.50
        ) {

            stuckCount++;
        }
    }
}


/* =========================================================
   VISUAL UPDATE
   ========================================================= */

function updateMeshes() {

    for (const unit of units) {

        if (!unit.mesh)
            continue;


        unit.mesh.position.set(
            unit.x,
            0,
            unit.z
        );


        /*
         * Slight visual cue:
         * attackers become brighter.
         */

        if (
            !unit.commander &&
            unit.attackCommand
        ) {

            unit.body.material.emissive =
                new THREE.Color(
                    0x222222
                );

        } else {

            unit.body.material.emissive =
                new THREE.Color(
                    0x000000
                );
        }
    }
}


/* =========================================================
   REWARD
   ========================================================= */

let redReward = 0;
let blueReward = 0;
let episodeReward = 0;


function calculateTimeoutReward() {

    const redAlive =
        units.filter(
            u =>
                u.team === RED &&
                !u.commander &&
                u.alive
        ).length;


    const blueAlive =
        units.filter(
            u =>
                u.team === BLUE &&
                !u.commander &&
                u.alive
        ).length;


    /*
     * Difference normalized by 100.
     */

    redReward =
        (redAlive - blueAlive) /
        SOLDIER_COUNT;

    blueReward =
        -redReward;

    episodeReward =
        redReward;

}


function calculateCommanderReward(
    winner
) {

    if (winner === "RED") {

        redReward = 1.0;
        blueReward = -1.0;

    } else {

        redReward = -1.0;
        blueReward = 1.0;
    }

    episodeReward =
        redReward;
}


/* =========================================================
   BATTLE END
   ========================================================= */

let battleEnded = false;


function endBattle(
    winner,
    reason
) {

    if (battleEnded)
        return;

    battleEnded = true;


    if (
        reason ===
        "commander"
    ) {

        calculateCommanderReward(
            winner
        );

    } else {

        calculateTimeoutReward();
    }


    const message =
        document.getElementById(
            "message"
        );


    let text = "";


    if (
        reason ===
        "commander"
    ) {

        text +=
            winner +
            " WIN<br>";

        text +=
            "Enemy commander defeated";

    } else {

        text +=
            "TIMEOUT<br>";

        text +=
            "Red reward: " +
            redReward.toFixed(3) +
            "<br>";

        text +=
            "Blue reward: " +
            blueReward.toFixed(3);
    }


    message.innerHTML =
        text;

    message.style.display =
        "block";


    console.log(
        "Battle finished",
        {
            winner,
            reason,
            redReward,
            blueReward,
            episodeReward
        }
    );
}


/* =========================================================
   DEBUG UI
   ========================================================= */

function countAlive(
    team
) {

    return units.filter(
        u =>
            u.team === team &&
            !u.commander &&
            u.alive
    ).length;
}


function updateUI(
    elapsed
) {

    document.getElementById(
        "time"
    ).textContent =
        elapsed.toFixed(1);


    document.getElementById(
        "remaining"
    ).textContent =
        Math.max(
            0,
            MAX_BATTLE_TIME - elapsed
        ).toFixed(1);


    document.getElementById(
        "redAlive"
    ).textContent =
        countAlive(RED);


    document.getElementById(
        "blueAlive"
    ).textContent =
        countAlive(BLUE);


    document.getElementById(
        "redCommanderHP"
    ).textContent =
        redCommander.hp.toFixed(2);


    document.getElementById(
        "blueCommanderHP"
    ).textContent =
        blueCommander.hp.toFixed(2);


    document.getElementById(
        "attacks"
    ).textContent =
        attackCount;


    document.getElementById(
        "blocked"
    ).textContent =
        blockedMoves;


    document.getElementById(
        "stuck"
    ).textContent =
        stuckCount;


    document.getElementById(
        "overlap"
    ).textContent =
        overlapCorrections;


    document.getElementById(
        "commandUpdates"
    ).textContent =
        commandUpdateCount;


    document.getElementById(
        "forwardPasses"
    ).textContent =
        forwardPassCount;


    document.getElementById(
        "inputSize"
    ).textContent =
        INPUT_SIZE;


    document.getElementById(
        "outputSize"
    ).textContent =
        OUTPUT_SIZE;


    document.getElementById(
        "redReward"
    ).textContent =
        redReward.toFixed(3);


    document.getElementById(
        "blueReward"
    ).textContent =
        blueReward.toFixed(3);
}


/* =========================================================
   RESIZE
   ========================================================= */

window.addEventListener(
    "resize",
    () => {

        camera.aspect =
            window.innerWidth /
            window.innerHeight;

        camera.updateProjectionMatrix();

        renderer.setSize(
            window.innerWidth,
            window.innerHeight
        );
    }
);


/* =========================================================
   MAIN LOOP
   ========================================================= */

let previousTime =
    performance.now();

let battleTime =
    0;

let commandAccumulator =
    COMMAND_INTERVAL;

let fpsTimer = 0;
let fpsFrames = 0;
let currentFPS = 0;


function animate() {

    requestAnimationFrame(
        animate
    );


    const now =
        performance.now();

    let dt =
        (now - previousTime) /
        1000.0;

    previousTime =
        now;


    /*
     * Protect against very large
     * time steps after tab switching.
     */

    dt =
        Math.min(
            dt,
            0.05
        );


    if (!battleEnded) {

        battleTime += dt;


        /*
         * NN commands are updated
         * every 0.20 sec.
         */

        commandAccumulator += dt;


        while (
            commandAccumulator >=
            COMMAND_INTERVAL
        ) {

            updateCommands();

            commandAccumulator -=
                COMMAND_INTERVAL;
        }


        updateMovement(dt);

        performAttacks(dt);

        resolveOverlaps();

        updateStuckDetection(dt);

        updateMeshes();


        /*
         * Timeout
         */

        if (
            battleTime >=
            MAX_BATTLE_TIME
        ) {

            endBattle(
                null,
                "timeout"
            );
        }
    }


    /*
     * FPS
     */

    fpsTimer += dt;
    fpsFrames++;


    if (
        fpsTimer >= 0.5
    ) {

        currentFPS =
            fpsFrames /
            fpsTimer;

        fpsTimer = 0;
        fpsFrames = 0;

        document.getElementById(
            "fps"
        ).textContent =
            currentFPS.toFixed(1);
    }


    updateUI(
        battleTime
    );


    renderer.render(
        scene,
        camera
    );
}


/* =========================================================
   INITIAL UI
   ========================================================= */

document.getElementById(
    "inputSize"
).textContent =
    INPUT_SIZE;


document.getElementById(
    "outputSize"
).textContent =
    OUTPUT_SIZE;


/* =========================================================
   START
   ========================================================= */

updateCommands();

animate();

</script>
</body>
</html>
'
></iframe>
"""

display(HTML(html))